In [24]:
# Importación de Librerías
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns 
from wordcloud import WordCloud
import os
import re




%matplotlib inline
pd.set_option('display.max_columns', 30)
plt.rcParams['figure.figsize'] = [12.0, 8.0]

In [27]:
dataset= pd.read_csv("input/petfinder-adoption-prediction/train/train.csv")
print("Directorio actual:", os.getcwd())
dataset.head(20)
print("Número de filas y columnas:", dataset.shape)

Directorio actual: c:\Users\Gloria\Documents\AUSTRAL\LAB2\UA_MDM_Labo2_Grupo12
Número de filas y columnas: (14993, 24)


In [28]:
train.dtypes

Type               int64
Name              object
Age                int64
Breed1             int64
Breed2             int64
Gender             int64
Color1             int64
Color2             int64
Color3             int64
MaturitySize       int64
FurLength          int64
Vaccinated         int64
Dewormed           int64
Sterilized         int64
Health             int64
Quantity           int64
Fee                int64
State              int64
RescuerID         object
VideoAmt           int64
Description       object
PetID             object
PhotoAmt         float64
AdoptionSpeed      int64
dtype: object

In [29]:
dataset.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14993 entries, 0 to 14992
Data columns (total 24 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           14993 non-null  int64  
 1   Name           13728 non-null  object 
 2   Age            14993 non-null  int64  
 3   Breed1         14993 non-null  int64  
 4   Breed2         14993 non-null  int64  
 5   Gender         14993 non-null  int64  
 6   Color1         14993 non-null  int64  
 7   Color2         14993 non-null  int64  
 8   Color3         14993 non-null  int64  
 9   MaturitySize   14993 non-null  int64  
 10  FurLength      14993 non-null  int64  
 11  Vaccinated     14993 non-null  int64  
 12  Dewormed       14993 non-null  int64  
 13  Sterilized     14993 non-null  int64  
 14  Health         14993 non-null  int64  
 15  Quantity       14993 non-null  int64  
 16  Fee            14993 non-null  int64  
 17  State          14993 non-null  int64  
 18  Rescue

In [30]:
train.Description


0        Nibble is a 3+ month old ball of cuteness. He ...
1        I just found it alone yesterday near my apartm...
2        Their pregnant mother was dumped by her irresp...
3        Good guard dog, very alert, active, obedience ...
4        This handsome yet cute boy is up for adoption....
                               ...                        
14988    I have 4 kittens that need to be adopt urgentl...
14989    Serato(female cat- 3 color) is 4 years old and...
14990    Mix breed, good temperament kittens. Love huma...
14991    she is very shy..adventures and independent..s...
14992    Fili just loves laying around and also loves b...
Name: Description, Length: 14993, dtype: object

In [33]:
char_feats = [f for f in dataset.columns if dataset[f].dtype=='O']
numeric_feats = [f for f in dataset.columns if dataset[f].dtype!='O']

In [35]:
char_feats

['Name', 'RescuerID', 'Description', 'PetID']

In [36]:
#preprocesamiento de texto para la columna description

def clean_text(text):
    if pd.isna(text):
        return ''
    # Convertir a minúsculas
    text = re.sub(r'\W+', ' ', text)  # Eliminar caracteres especiales
    return text


In [37]:
dataset['Description_limpia'] = dataset['Description'].apply(clean_text)

In [38]:
dataset['Description_limpia'].head(10)

0    Nibble is a 3 month old ball of cuteness He is...
1    I just found it alone yesterday near my apartm...
2    Their pregnant mother was dumped by her irresp...
3    Good guard dog very alert active obedience wai...
4    This handsome yet cute boy is up for adoption ...
5    This is a stray kitten that came to my house H...
6    anyone within the area of ipoh or taiping who ...
7    Siu Pak just give birth on 13 6 10 to 6puppies...
8    healthy and active feisty kitten found in neig...
9    Very manja and gentle stray cat found we would...
Name: Description_limpia, dtype: object

In [39]:
# Función para identificar caracteres especiales eliminados
def get_removed_special_chars(original, cleaned):
    if pd.isna(original):  # Si el texto original es NaN
        return []
    # Convertir ambos textos a conjuntos de caracteres
    original_chars = set(original)  # Conjunto de caracteres del texto original
    cleaned_chars = set(cleaned)    # Conjunto de caracteres del texto limpio
    # Identificar los caracteres que están en el original pero no en el limpio
    removed_chars = original_chars - cleaned_chars
    # Filtrar solo los caracteres especiales (no alfanuméricos)
    special_chars = [char for char in removed_chars if not char.isalnum() and not char.isspace()]
    return special_chars

# Aplicar la función para generar la nueva columna
dataset['Removed_Special_Chars'] = dataset.apply(
    lambda row: get_removed_special_chars(row['Description'], row['Description_limpia']), axis=1
)

In [42]:
descripcion_df=dataset[['Description', 'Description_limpia', 'Removed_Special_Chars']]
descripcion_df.head(5)

,Description,Description_limpia,Removed_Special_Chars
0,Nibble is a 3+ month old ball of cuteness. He ...,Nibble is a 3 month old ball of cuteness He is...,"[., +, ']"
1,I just found it alone yesterday near my apartm...,I just found it alone yesterday near my apartm...,[.]
2,Their pregnant mother was dumped by her irresp...,Their pregnant mother was dumped by her irresp...,"[., ,]"
3,"Good guard dog, very alert, active, obedience ...",Good guard dog very alert active obedience wai...,"[!, ,]"
4,This handsome yet cute boy is up for adoption....,This handsome yet cute boy is up for adoption ...,"[., ,, ']"


In [47]:
X = dataset.drop('AdoptionSpeed', axis=1)
y = dataset['AdoptionSpeed']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)